In [ ]:
import os
from pathlib import Path

import pandas as pd

dataset_path = Path(
    os.environ.get(
        "SIS_DATASET_PATH",
        str(Path.home() / "OneDrive" / "student_habits_performance.csv"),
    )
)
df = pd.read_csv(dataset_path)


In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.head(20)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sns.set(style="whitegrid")

In [ ]:
df.isna()

In [ ]:
df.isna().sum()

In [ ]:
df.isna().sum().sum()

In [ ]:
df.dropna()

In [ ]:
df.dropna(inplace=True)

In [ ]:
df

In [ ]:
df.duplicated().sum()

In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
df.describe()

In [ ]:
df.describe(include="object").columns

In [ ]:
categorical_cols = [
    "gender",
    "part_time_job",
    "diet_quality",
    "parental_education_level",
    "internet_quality",
    "extracurricular_participation",
]

In [ ]:
for col in categorical_cols:
    print(f"Value counts for {col}: \n {df[col].value_counts()}")

In [ ]:
df.hist(bins=20, edgecolor="black")
plt.tight_layout()
plt.show()

In [ ]:
for col in categorical_cols:
    sns.countplot(data=df, x=col)
    plt.title(f"Distribution of {col}")
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
df.corr(numeric_only=True)

In [ ]:
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation matrix")
plt.show()

In [ ]:
df.describe().columns

In [ ]:
num_features = [
    "age",
    "study_hours_per_day",
    "social_media_hours",
    "netflix_hours",
    "attendance_percentage",
    "sleep_hours",
    "exercise_frequency",
    "mental_health_rating",
]

In [ ]:
for feature in num_features:
    sns.scatterplot(data=df, x=feature, y="exam_score")
    plt.title(f"{feature} vs Exam Score")
    plt.show()

In [ ]:
for col in categorical_cols:
    sns.boxplot(data=df, x=col, y="exam_score")
    plt.title(f"Exam Score by {col}")
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

In [ ]:
df.columns

In [ ]:
df.head(2)

In [ ]:
features = [
    "study_hours_per_day",
    "attendance_percentage",
    "mental_health_rating",
    "sleep_hours",
    "part_time_job",
]


In [ ]:
target = "exam_score"

In [ ]:
df_model = df[features + [target]].copy()

In [ ]:
df_model

In [ ]:
le = LabelEncoder()

In [ ]:
df_model["part_time_job"] = le.fit_transform(df_model["part_time_job"])

In [ ]:
df_model

In [ ]:
x = df_model[features].rename(
    columns={
        "study_hours_per_day": "study_hours",
        "attendance_percentage": "attendance",
        "mental_health_rating": "mental_health",
    }
)


In [ ]:
y = df_model[target]

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)


In [ ]:
len(y_test)

In [ ]:
len(y_train)

In [ ]:
models = {
    "Linear Regression": {"model": LinearRegression(), "params": {}},
    "DecisionTree": {
        "model": DecisionTreeRegressor(random_state=42),
        "params": {"max_depth": [3, 5, 10], "min_samples_split": [2, 5]},
    },
    "RandomForest": {
        "model": RandomForestRegressor(random_state=42),
        "params": {"n_estimators": [50, 100], "max_depth": [5, 10]},
    },
}


In [ ]:
best_models = []

In [ ]:
for name, confiq in models.items():
    print(f"Training {name}")

    grid = GridSearchCV(
        confiq["model"], confiq["params"], cv=5, scoring="neg_mean_squared_error"
    )
    grid.fit(x_train, y_train)

    y_pred = grid.predict(x_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    best_models.append(
        {"model": name, "best_params": grid.best_params_, "rmse": rmse, "r2 ": r2}
    )

In [ ]:
best_models

In [ ]:
results_df = pd.DataFrame(best_models)

In [ ]:
results_df.sort_values(by="rmse")

In [ ]:
import joblib

best_row = results_df.sort_values(by="rmse").iloc[0]

In [ ]:
best_row

In [ ]:
best_model_name = best_row["model"]

In [ ]:
best_model_name

In [ ]:
best_model_config = models[best_model_name]

In [ ]:
best_model_config

In [ ]:
final_model = best_model_config["model"].set_params(**best_row["best_params"])


In [ ]:
final_model.fit(x, y)


In [ ]:
final_model.predict(x_test)


In [ ]:
project_root = Path.cwd()
if not (project_root / "app.py").is_file():
    project_root = project_root.parent

model_output = project_root / "models" / "best_model.pkl"
model_output.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(final_model, model_output)
print(f"Saved compatible model to {model_output}")


In [ ]:
joblib.load(model_output).predict(x_test)
